# Семинар 2. Сегментация клиентов: RFM-анализ и K-Means кластеризация

**Цель семинара:** Научиться превращать «сырой» транзакционный лог в структурированные бизнес-сегменты клиентов с помощью RFM-анализа и алгоритма машинного обучения K-Means без учителей. 

Результат этого семинара — колонка `Cluster_ID` в таблице клиентов — станет одним из ключевых признаков обогащения данных для прогнозных моделей ИИ на последующих занятиях.

---

## Зачем вообще сегментировать клиентов?
Представьте, что вы управляете крупным бизнесом. У вас 1000 клиентов, и вы хотите отправить им персональное предложение. Но у вас нет ресурсов, чтобы вручную придумать 1000 уникальных маркетинговых стратегий.

**Решение:** Разбить клиентов на несколько однородных групп по их фактическому поведению и сделать таргетированные предложения. Клиентам, которые давно не покупали — скидку на реактивацию. Лояльным «чемпионам» — доступ к закрытой программе лояльности. «Спящим» — легкое напоминание о новинках.

Именно это и делает связка RFM + K-Means: никакой магии — только три цифры на клиента и математика расстояний.

---

## Что такое RFM?

RFM — это три метрики, описывающие операционную активность клиента с разных ракурсов:

| Буква | Название | Вопрос бизнес-анализа | Пример |
|-------|----------|------------------------|--------|
| **R** | Recency (Давность) | Сколько дней прошло с **последней** покупки? | 5 дней = активный; 200 дней = критический риск оттока |
| **F** | Frequency (Частота) | Сколько **раз** за весь период совершались операции? | 120 раз = высокая лояльность; 2 раза = случайный визит |
| **M** | Monetary (Деньги) | Какова **суммарная** финансовая ценность клиента? | 6000 руб. = VIP; 800 руб. = низкомаржинальный сегмент |

Идея простая: идеальный клиент покупал недавно (R↓), часто (F↑) и тратил много (M↑).


## 🧰 Шаг 0. Импорт библиотек и настройка окружения

Сегодня нам понадобится расширенный стек библиотек для анализа данных и машинного обучения:
- **pandas / numpy** — манипуляции с массивами и таблицами.
- **matplotlib / seaborn** — отрисовка графиков и профилей кластеров.
- **sklearn.preprocessing.StandardScaler** — нормализация признаков (выравнивание масштабов).
- **sklearn.cluster.KMeans** — промышленный алгоритм кластеризации.
- **sklearn.decomposition.PCA** — сжатие размерности для 2D-визуализации многомерных пространств.


In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')

print("✅ Все библиотеки успешно загружены!")
print("📦 Версии: pandas", pd.__version__, "| scikit-learn готов к кластеризации")


## 📥 Шаг 1. Локализация и загрузка транзакционного лога

Данные вашего варианта выгружены в локальную директорию `data/input/transactions.csv`. Это классическая «таблица фактов» (Facts): на одного клиента (`Target_ID`) приходится множество строк, фиксирующих историю его действий. Наша задача — сжать этот лог в компактный профиль клиента (Dimensions).


In [ ]:
INPUT_DIR = os.path.join(".", "data", "input")
DATA_PATH = os.path.abspath(os.path.join(INPUT_DIR, "transactions.csv"))

if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Критическая ошибка: файл не обнаружен по пути: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
print(f"📊 Загружено строк истории: {df.shape[0]:,}")
print(f"👥 Уникальных клиентов в логе: {df['Target_ID'].nunique()}")


---

## 🛠 ЗАДАНИЕ 1: Рентген транзакционных данных
**Задача:** Проведите экспресс-диагностику загруженной таблицы фактов. Выведите типы данных (`.info()`), описательную статистику трат (`.describe()`) и проверьте таблицу на наличие явных пропусков.

*🤖 Возникли сложности? Используйте теги для помощи ИИ-тьютора:*
* `#SEM2_TASK1_START` — если забыли синтаксис методов проверки пропусков.
* `#SEM2_TASK1_WHY` — если хотите обсудить, почему в таблице фактов нельзя удалять строки с повторяющимися Target_ID.


In [ ]:
# [MASTER SOLUTION]
print("--- ПАСПОРТ ТАБЛИЦЫ ---")
df.info()

print("\n--- БАЗОВАЯ СТАТИСТИКА ---")
print(df.describe())

print("\n--- ПРОПУСКИ ---")
print(df.isnull().sum())


In [ ]:
# [STUDENT TEMPLATE]
# TODO: Вызовите методы .info(), .describe() и проверьте наличие пропусков (.isnull().sum())
raise NotImplementedError("Задание 1 не выполнено! Удалите эту строку и напишите свой код.")

print("--- ПАСПОРТ ТАБЛИЦЫ ---")
# Ваш код для info()

print("\n--- БАЗОВАЯ СТАТИСТИКА ---")
# Ваш код для describe()

print("\n--- ПРОПУСКИ ---")
# Проверьте наличие пропусков


---

## 🛠 ЗАДАНИЕ 2: Починим дату и определим «точку отсчёта»
**Задача:** Временная метка `Trans_Date` считалась как текстовая строка (`object`). Переведите её в формат `datetime`. Найдите максимальную (самую свежую) дату транзакции в датасете и запишите её в переменную `snapshot_date` — она станет нашей точкой «сегодня» для расчёта давности.

*🤖 Помощь AI-тьютора:*
* `#SEM2_TASK2_START` — синтаксис `pd.to_datetime`.
* `#SEM2_TASK2_BUG` — ошибка парсинга временных строк (формат даты).


In [ ]:
# [MASTER SOLUTION]
df['Trans_Date'] = pd.to_datetime(df['Trans_Date'])
snapshot_date = df['Trans_Date'].max()

print(f"📅 Точка отсчёта (snapshot): {snapshot_date}")
print(f"📅 Тип данных Trans_Date: {df['Trans_Date'].dtype}")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: 1. Преобразуйте Trans_Date в datetime
# TODO: 2. Найдите максимальную дату в датасете и сохраните в snapshot_date
raise NotImplementedError("Задание 2 не выполнено! Удалите эту строку и напишите свой код.")

# Преобразуйте колонку Trans_Date в datetime:
df['Trans_Date'] = ...

# Найдите максимальную дату и запишите в snapshot_date:
snapshot_date = ...

print(f"📅 Точка отсчёта (snapshot): {snapshot_date}")
print(f"📅 Тип данных Trans_Date: {df['Trans_Date'].dtype}")


---

## 🛠 ЗАДАНИЕ 3: Вычисляем агрегированные RFM-метрики
**Задача:** Сгруппируйте таблицу по `Target_ID` и сожмите историю транзакций до уровня уникальных клиентов, рассчитав:
- `Recency`: разность в днях между `snapshot_date` и датой **последней** (максимальной) покупки клиента.
- `Frequency`: общее **количество** транзакций клиента за весь период.
- `Monetary`: **суммарный объём** всех трат клиента (`Trans_Amount`).

Сбросьте индекс (`.reset_index()`), сохраните таблицу в переменную `rfm` и выведите первые 5 строк.

*🤖 Помощь AI-тьютора:*
* `#SEM2_TASK3_START` — шаблон для `.groupby().agg()` с использованием lambda-функции для вычисления дней Recency.
* `#SEM2_TASK3_BUG` — если после агрегации в таблице не осталось колонок с именами метрик.
* `#SEM2_TASK3_WHY` — разбор бизнес-смысла распределения метрик в `.describe()`.


In [ ]:
# [MASTER SOLUTION]
rfm = df.groupby('Target_ID').agg(
    Recency  = ('Trans_Date',   lambda x: (snapshot_date - x.max()).days),
    Frequency= ('Trans_Date',   'count'),
    Monetary = ('Trans_Amount', 'sum')
).reset_index()

print("📈 Статистика RFM-профилей клиентов:")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(1))


In [ ]:
# [STUDENT TEMPLATE]
# TODO: Сгруппируйте данные по Target_ID и вычислите метрики Recency, Frequency, Monetary
raise NotImplementedError("Задание 3 не выполнено! Удалите эту строку и напишите свой код.")

rfm = df.groupby('Target_ID').agg(
    Recency  = ('Trans_Date',   ...),   # Примените lambda x: (snapshot_date - x.max()).days
    Frequency= ('Trans_Date',   ...),   # Подсчитайте количество транзакций
    Monetary = ('Trans_Amount', ...)    # Посчитайте сумму трат
).reset_index()

print("📈 Статистика RFM-профилей клиентов:")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(1))


---

## 🛠 ЗАДАНИЕ 4: Нормализация — выравниваем масштабы признаков
**Задача:** Обратите внимание на дисбаланс масштабов признаков: Recency измеряется в днях (0–270), Frequency в штуках (до 150), а Monetary в тысячах рублей. Алгоритм K-Means основан на расчёте евклидовых расстояний, поэтому Monetary начнет доминировать над Recency просто из-за своего масштаба чисел. 

Используйте класс `StandardScaler`, чтобы привести все три метрики к единому масштабу со средним $\approx 0$ и стандартным отклонением $= 1$. Результат запишите в матрицу `rfm_scaled`.

*🤖 Помощь AI-тьютора:*
* `#SEM2_TASK4_START` — как вызвать методы `.fit_transform()` у инстанса скалера.
* `#SEM2_TASK4_WHY` — почему без стандартизации алгоритмы кластеризации и линейные модели ИИ теряют свою предсказательную силу.


In [ ]:
# [MASTER SOLUTION]
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

print(f"✅ Нормализация выполнена! Форма массива: {rfm_scaled.shape}")
print("Первые 3 строки нормализованной матрицы:\n", rfm_scaled[:3].round(3))


In [ ]:
# [STUDENT TEMPLATE]
# TODO: Инициализируйте StandardScaler и нормализуйте три колонки ['Recency', 'Frequency', 'Monetary']
raise NotImplementedError("Задание 4 не выполнено! Удалите эту строку и напишите свой код.")

scaler = ...
rfm_scaled = ...

print(f"✅ Нормализация выполнена! Форма массива: {rfm_scaled.shape}")
print("Первые 3 строки нормализованной матрицы:\n", rfm_scaled[:3].round(3))


---

## 🛠 ЗАДАНИЕ 5: Метод «локтя» — математический выбор количества кластеров
**Задача:** Машине нужно подсказать параметр количества кластеров `k`. Реализуйте цикл `for k in range(2, 11)`, на каждой итерации обучая модель `KMeans(n_clusters=k, random_state=42, n_init=10)` на нормализованных данных `rfm_scaled`. Сохраняйте показатель внутрикластерной инерции модели (`km.inertia_`) в список `inertias`. Изучите построенный график и найдите точку перелома ("локоть").

*🤖 Помощь AI-тьютора:*
* `#SEM2_TASK5_START` — структура цикла обучения и метод `.append()` для накопления инерции.
* `#SEM2_TASK5_WHY` — как математически интерпретировать график инерции и почему бесконечно увеличивать k бессмысленно.


In [ ]:
# [MASTER SOLUTION]
inertias = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(rfm_scaled)
    inertias.append(km.inertia_)

# Отрисовка графика метода локтя
plt.figure(figsize=(8, 4))
plt.plot(range(2, 11), inertias, marker='o', color='steelblue', linewidth=2)
plt.title('Метод локтя: выбор оптимального k', fontsize=12, fontweight='bold')
plt.xlabel('Количество кластеров (k)')
plt.ylabel('Инерция (Inertia)')
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# [STUDENT TEMPLATE]
# TODO: В цикле переберите k от 2 до 10, обучите KMeans и сохраните km.inertia_ в список inertias
raise NotImplementedError("Задание 5 не выполнено! Удалите эту строку и напишите свой код.")

inertias = []
for k in range(2, 11):
    km = KMeans(n_clusters=..., random_state=42, n_init=10)
    # Обучите модель:
    ...
    # Сохраните инерцию:
    inertias.append(...)

# --- График отрисовки (не меняйте) ---
plt.figure(figsize=(8, 4))
plt.plot(range(2, 11), inertias, marker='o', color='steelblue', linewidth=2)
plt.title('Метод локтя: выбор оптимального k', fontsize=12, fontweight='bold')
plt.xlabel('Количество кластеров (k)')
plt.ylabel('Инерция (Inertia)')
plt.grid(True, alpha=0.3)
plt.show()


---

## 🛠 ЗАДАНИЕ 6: Обучение финальной модели кластеризации
**Задача:** График локтя показывает явное выравнивание падения инерции после отметки 4. Инициализируйте и обучите итоговую модель `KMeans` с параметром `n_clusters=4`. Извлеките метки кластеров через свойство `.labels_` и запишите их в исходную таблицу `rfm` в виде нового категориального признака `Cluster_ID`.

*🤖 Помощь AI-тьютора:*
* `#SEM2_TASK6_START` — запуск финального обучения с фиксацией `random_state=42`.
* `#SEM2_TASK6_BUG` — ошибка несовпадения размерностей при записи меток в DataFrame.


In [ ]:
# [MASTER SOLUTION]
kmeans_final = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm['Cluster_ID'] = kmeans_final.fit_predict(rfm_scaled)

print("📊 Распределение клиентов по полученным кластерам:")
print(rfm['Cluster_ID'].value_counts().sort_index())


In [ ]:
# [STUDENT TEMPLATE]
# TODO: Обучите KMeans на k=4 и добавьте колонку 'Cluster_ID' в датафрейм rfm
raise NotImplementedError("Задание 6 не выполнено! Удалите эту строку и напишите свой код.")

kmeans_final = KMeans(n_clusters=..., random_state=42, n_init=10)
# Обучите модель и получите метки кластеров:
rfm['Cluster_ID'] = ...

print("📊 Распределение клиентов по полученным кластерам:")
print(rfm['Cluster_ID'].value_counts().sort_index())


---

## 📊 Шаг 7. Профилирование поведенческих сегментов

Метки кластеров от 0 до 3 абстрактны. Чтобы превратить их в бизнес-знания, рассчитаем средние значения RFM для каждого кластера. Этот код уже написан за вас. Изучите выведенную матрицу профилей и столбчатые диаграммы.


In [ ]:
# Расчет средних профилей кластеров
cluster_profile = rfm.groupby('Cluster_ID')[['Recency', 'Frequency', 'Monetary']].mean().round(1)
cluster_profile['Размер_Кластера'] = rfm['Cluster_ID'].value_counts().sort_index()

print("📊 Портрет кластеров в реальных бизнес-метриках:")
print(cluster_profile)

# Визуализация портретов сегментов
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['Recency', 'Frequency', 'Monetary']
colors = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

for i, metric in enumerate(metrics):
    vals = cluster_profile[metric]
    axes[i].bar(vals.index.astype(str), vals.values, color=colors, alpha=0.8)
    axes[i].set_title(f'Средний {metric}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Кластер')
    for j, v in enumerate(vals.values):
        axes[i].text(j, v * 1.01, f'{v:.0f}', ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Сравнительный анализ поведенческих характеристик кластеров', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()


---

## 🛠 ЗАДАНИЕ 7: Маркетинговая интерпретация и присвоение бизнес-имен
**Задача:** Проанализируйте график и таблицу шага 7. Дайте каждому кластеру от 0 до 3 осмысленное имя (например, *"VIP / Чемпионы"*, *"Спящие"*, *"Новички"*, *"Уходящие / Группа риска"*), опираясь на логику метрик (кто покупает часто и много, а кто давно пропал). Назовите сегменты и запишите их в новую колонку `Segment` через метод `.map()`.

*🤖 Помощь AI-тьютора:*
* `#SEM2_TASK7_WHY` — если сомневаетесь, какому именно кластеру подходит конкретное название. Назовите тьютору средние числа вашего вывода, и он поможет сопоставить их с маркетинговыми архетипами.


In [ ]:
# [MASTER SOLUTION]
# Пример маппинга (имена могут варьироваться, главное — уйти от плейсхолдера '...')
segment_names = {
    0: "Спящие клиенты",
    1: "Лояльные активные",
    2: "VIP / Чемпионы",
    3: "Группа риска / Отток"
}

rfm['Segment'] = rfm['Cluster_ID'].map(segment_names)
print("✅ Бизнес-имена успешно присвоены. Топ-5 записей витрины:")
print(rfm[['Target_ID', 'Recency', 'Frequency', 'Monetary', 'Cluster_ID', 'Segment']].head())


In [ ]:
# [STUDENT TEMPLATE]
# TODO: Изучите средние значения кластеров выше и замените '...' на текстовые имена сегментов
raise NotImplementedError("Задание 7 не выполнено! Удалите эту строку и напишите свой код.")

segment_names = {
    0: "...",
    1: "...",
    2: "...",
    3: "..."
}

# Примените словарь segment_names к колонке Cluster_ID через метод .map()
rfm['Segment'] = ...

print(rfm[['Target_ID', 'Recency', 'Frequency', 'Monetary', 'Cluster_ID', 'Segment']].head())


---

## 📊 Шаг 8. Проекция многомерного пространства кластеров в 2D (PCA)

Поскольку наше RFM пространство трехмерно, для визуализации на плоскости мы используем алгоритм снижения размерности PCA (Метод главных компонент). Он сжимает 3 колонки в 2 новые латентные компоненты, сохраняя максимум дисперсии. Данный код является демонстрационным, просто запустите ячейку.


In [ ]:
pca = PCA(n_components=2, random_state=42)
rfm_2d = pca.fit_transform(rfm_scaled)

rfm_plot = rfm.copy()
rfm_plot['PC1'] = rfm_2d[:, 0]
rfm_plot['PC2'] = rfm_2d[:, 1]

plt.figure(figsize=(10, 6))
palette = ['#E74C3C', '#3498DB', '#2ECC71', '#F39C12']

for cid in sorted(rfm_plot['Cluster_ID'].unique()):
    mask = rfm_plot['Cluster_ID'] == cid
    label = f"Кластер {cid}: {rfm_plot.loc[mask, 'Segment'].iloc[0]}"
    plt.scatter(rfm_plot.loc[mask, 'PC1'], rfm_plot.loc[mask, 'PC2'], label=label, alpha=0.7, s=50, color=palette[cid])

plt.title('Визуализация поведенческих сегментов в 2D пространстве (PCA)', fontsize=12, fontweight='bold')
plt.xlabel('Главная Компонента 1 (PC1)')
plt.ylabel('Главная Компонента 2 (PC2)')
plt.legend(fontsize=9, loc='best')
plt.grid(True, alpha=0.2)
plt.show()


---

## 🛠 ЗАДАНИЕ 8: Экспорт аналитической витрины для ИИ-моделей
**Задача:** Сохраните итоговый датафрейм `rfm` в локальный файл `data/input/rfm_with_clusters.csv` с помощью метода `.to_csv()`. Экспортируйте данные строго **без** технического индекса строк (`index=False`), чтобы не засорять будущие обучающие выборки.

*🤖 Помощь AI-тьютора:*
* `#SEM2_TASK8_START` — синтаксис корректного вызова `.to_csv()` с флагом отключения индексов.


In [ ]:
# [MASTER SOLUTION]
OUTPUT_PATH = os.path.join(INPUT_DIR, "rfm_with_clusters.csv")
rfm.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Файл успешно сохранен для этапа ML-моделирования: {OUTPUT_PATH}")


In [ ]:
# [STUDENT TEMPLATE]
# TODO: Сохраните таблицу rfm по пути OUTPUT_PATH, отключив индекс (index=False)
raise NotImplementedError("Задание 8 не выполнено! Удалите эту строку и напишите свой код.")

OUTPUT_PATH = os.path.join(INPUT_DIR, "rfm_with_clusters.csv")

# Сохраните результат в CSV:
...

print(f"✅ Проверка: Файл успешно создан.")


---

## 🚀 Автоматизированная проверка качества (Autocheck)

Запустите данный скрипт валидации. Он проверяет соответствие полученной аналитической RFM-витрины жестким техническим и архитектурным критериям программы МИИБА.


In [ ]:
def run_autocheck():
    print("🚀 Запуск автоматической проверки результатов Семинара 2...\n" + "-" * 50)
    passed = True

    # 1. Проверка структуры лога
    try:
        if not isinstance(df, pd.DataFrame):
            print("❌ Ошибка: Переменная 'df' потеряла тип DataFrame.")
            passed = False
        else:
            print(f"✅ Лог транзакций валиден ({df.shape[0]:,} строк).")
    except NameError:
        print("❌ Ошибка: Базовая таблица 'df' не найдена.")
        passed = False

    # 2. Проверка агрегации RFM
    try:
        if rfm.shape[0] != 1000:
            print(f"❌ Ошибка: В RFM-таблице {rfm.shape[0]} строк. Ожидалось ровно 1000 клиентов.")
            passed = False
        elif not all(col in rfm.columns for col in ['Recency', 'Frequency', 'Monetary']):
            print("❌ Ошибка: В структуре rfm отсутствуют обязательные метрики.")
            passed = False
        elif rfm['Recency'].min() < 0:
            print("❌ Ошибка: Найдена отрицательная давность Recency. Проверьте вектор разности дат.")
            passed = False
        else:
            print(f"✅ RFM-витрина успешно агрегирована для всех {rfm.shape[0]} клиентов.")
    except NameError:
        print("❌ Ошибка: Результирующий датафрейм 'rfm' не инициализирован.")
        passed = False

    # 3. Проверка нормализации
    try:
        if rfm_scaled.shape != (1000, 3):
            print("❌ Ошибка: Неверная размерность масштабированной матрицы rfm_scaled.")
            passed = False
        elif abs(rfm_scaled.mean()) > 0.05:
            print("❌ Ошибка: Математическое среднее нормализованных данных слишком далеко от нуля.")
            passed = False
        else:
            print("✅ Масштабирование признаков StandardScaler выполнено корректно.")
    except NameError:
        print("❌ Ошибка: Нормализованный массив 'rfm_scaled' не найден.")
        passed = False

    # 4. Проверка кластеризации и бизнес-именований
    try:
        if 'Cluster_ID' not in rfm.columns or 'Segment' not in rfm.columns:
            print("❌ Ошибка: В таблице отсутствуют целевые колонки Cluster_ID или Segment.")
            passed = False
        elif rfm['Cluster_ID'].nunique() != 4:
            print(f"❌ Ошибка: Найдено {rfm['Cluster_ID'].nunique()} кластеров, требуется строго 4.")
            passed = False
        elif any(v.strip() == '...' for v in rfm['Segment'].unique()):
            print("❌ Ошибка: Обнаружены нераспознанные бизнес-плейсхолдеры '...'. Заполните словарь названий.")
            passed = False
        else:
            print(f"✅ Сегментация K-Means успешно зафиксирована: {rfm['Segment'].unique().tolist()}")
    except Exception as e:
        print(f"❌ Критическая ошибка при валидации сегментов: {e}")
        passed = False

    # 5. Проверка дискового экспорта
    export_check_path = os.path.join(INPUT_DIR, "rfm_with_clusters.csv")
    if not os.path.exists(export_check_path):
        print("❌ Ошибка: Финальный файл rfm_with_clusters.csv не обнаружен на диске.")
        passed = False
    else:
        print("✅ Аналитический артефакт успешно экспортирован в локальное хранилище.")

    print("-" * 50)
    if passed:
        print("🎉 ПОЗДРАВЛЯЕМ! Технический контур сегментации данных выполнен безупречно!")
        print("Внесите ваши финальные бизнес-выводы в аналитическую ячейку выше.")
    else:
        print("⚠️ Пайплайн содержит дефекты. Исправьте ошибки в ячейках и запустите Autocheck повторно.")

run_autocheck()